In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
import numpy as np
from pyspark.sql import Row
from pyspark.sql import SparkSession
from redis.cluster import RedisCluster, ClusterNode

In [2]:
memory_use = "16g"
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", memory_use)
    .config("spark.executor.memory", memory_use)
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")

    # .config("spark.redis.host", "172.25.0.10") # Node đầu tiên
    # .config("spark.redis.port", "6379")
    # .config("spark.redis.user", "default")
    # .config("spark.redis.auth", "1234")
    # Kích hoạt chế độ Cluster để Spark biết có 8 shards
    .config("spark.redis.cluster.enabled", "true") 
    .getOrCreate())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/07 12:59:35 WARN Utils: Your hostname, lenovo-Legion-5-15IAH7H, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp0s20f3)
26/05/07 12:59:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 12:59:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/07 12:59:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# Tạo danh sách các row_id cần lấy (từ 1 đến 10)
target_ids = range(1000, 1010)

# Tạo RDD từ danh sách IDs và thực hiện truy vấn trực tiếp vào Redis
def get_specific_rows(row_id):
    from redis.cluster import RedisCluster, ClusterNode
    
    startup_nodes = [ClusterNode("172.25.0.10", 6379)]
    rc = RedisCluster(startup_nodes=startup_nodes, password='1234', decode_responses=True)
    
    # Tạo key tương ứng
    redis_key = f"drug:{row_id}"
    
    # Lấy dữ liệu Hash
    data = rc.hgetall(redis_key)
    
    return data if data else None

# Thực thi song song và chuyển về DataFrame
# Chúng ta dùng 1 partition vì chỉ có 10 dòng, không cần chia nhỏ hơn
specific_rdd = spark.sparkContext.parallelize(target_ids, 1).map(get_specific_rows)

# Loại bỏ các dòng None (nếu Key không tồn tại trong Redis)
df_top_10 = specific_rdd.filter(lambda x: x is not None).toDF()

# Hiển thị kết quả
df_top_10.orderBy(F.col("row_id").cast("int")).show(truncate=False)

+---+------+-----------+-----+------------------+---+------+
|Age|BP    |Cholesterol|Drug |Na_to_K           |Sex|row_id|
+---+------+-----------+-----+------------------+---+------+
|49 |HIGH  |HIGH       |DrugA|12.887999534606934|F  |1000  |
|73 |LOW   |HIGH       |DrugC|10.050999641418457|M  |1001  |
|30 |NORMAL|NORMAL     |DrugX|13.138999938964844|F  |1002  |
|16 |LOW   |HIGH       |DrugY|32.82500076293945 |M  |1003  |
|51 |NORMAL|NORMAL     |DrugY|19.016000747680664|F  |1004  |
|39 |NORMAL|NORMAL     |DrugY|26.54199981689453 |F  |1005  |
|33 |NORMAL|NORMAL     |DrugY|28.091999053955078|F  |1006  |
|28 |HIGH  |HIGH       |DrugY|30.190000534057617|F  |1007  |
|29 |NORMAL|NORMAL     |DrugX|7.5320000648498535|F  |1008  |
|54 |NORMAL|HIGH       |DrugY|24.250999450683594|M  |1009  |
+---+------+-----------+-----+------------------+---+------+

